# Exploratory Data Analysis - Online Retail II

**Project:** Customer Segmentation & Retention Analytics  
**Author:** César Duarte  
**Dataset:** [Online Retail II — UCI ML Repository](https://archive.ics.uci.edu/dataset/502/online+retail+ii)  
**Status:** In Progress

## Table of Contents

1. [Introduction](#1-introduction)
2. [Setup & Data Loading](#2-setup--data-loading)
3. [Data Overview](#3-data-overview)
4. [Data Quality Assessment](#4-data-quality-assessment)
   - 4.1 Missing Values
   - 4.2 Cancellations
   - 4.3 Invalid Prices
   - 4.4 Negative Quantities Outside Cancellations
5. [Univariate Analysis](#5-univariate-analysis)
   - 5.1 Quantity
   - 5.2 Price
   - 5.3 Country
   - 5.4 Customer ID
6. [Temporal Analysis](#6-temporal-analysis)
   - 6.1 Monthly Revenue
   - 6.2 Daily and Weekly Patterns
7. [Customer Analysis](#7-customer-analysis)
   - 7.1 Purchases per Customer
   - 7.2 Revenue per Customer
   - 7.3 Top Customers
8. [Product Analysis](#8-product-analysis)
   - 8.1 Top Products by Revenue
   - 8.2 Top Products by Frequency
9. [Conclusions & Cleaning Decisions](#9-conclusions--cleaning-decisions)

---

## 1. Introduction <a id="1-introduction"></a>

### Project Description

**Goal:** Understand the raw transactional data before any cleaning or modeling. This notebook answers the question: *what is in this data, and what needs to be fixed before we can use it?*

**Questions to answer:**
- What is the overall shape and quality of the data?
- What are the main data quality issues (nulls, cancellations, invalid values)?
- What patterns exist in purchasing behavior over time?
- What does the customer base look like in terms of value and frequency?
- What decisions should guide the cleaning step?

### Dataset Description

| Column       | Type     | Description                                      |
|:-------------|:---------|:-------------------------------------------------|
| Invoice      | str      | Invoice number. Prefix 'C' = cancellation        |
| StockCode    | str      | Product code                                     |
| Description  | str      | Product description                              |
| Quantity     | int      | Units per transaction (negative = return)        |
| InvoiceDate  | datetime | Date and time of transaction                     |
| Price        | float    | Unit price in GBP                                |
| Customer ID  | str      | Unique customer identifier (NaN = guest)         |
| Country      | str      | Country of the customer                          |

### Pipeline Prerequisites

This notebook assumes the ingestion pipeline has **already been executed**:

```bash
pixi run python src/ingestion/loader.py
pixi run python src/ingestion/validation.py
```

These scripts read the raw Excel file, apply type normalization, validate data integrity, and save the processed Parquet to `data/processed/online_retail_ii.parquet`

---

## 2. Setup & Data Loading <a id="2-setup--data-loading"></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR = Path().resolve().parents[0]
PROCESSED_DIR = BASE_DIR / "data/processed"

In [2]:
# ── Processed Data ────────────────────────────────────────────────────────────
df = pd.read_parquet(PROCESSED_DIR / "online_retail_ii.parquet")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded: 1,067,371 rows x 8 columns


---

## 3. Data Overview <a id="3-data-overview"></a>

In [3]:
# First look
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [4]:
# Shape, dtypes, memory
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  str           
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   str           
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(1), int64(1), str(5)
memory usage: 121.4 MB


In [5]:
# Descriptive statistics for numeric columns
df.describe()

,Quantity,InvoiceDate,Price
count,1.067371e+06,1067371,1.067371e+06
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04
std,1.727058e+02,NaN,1.235531e+02


In [6]:
# Descriptive statistics for categorical columns
df.describe(include="str")

,Invoice,StockCode,Description,Customer ID,Country
count,1067371,1067371,1062989,824364,1067371
unique,53628,5305,5698,5942,43
top,537434,85123A,WHITE HANGING HEART T-LIGHT HOLDER,17841,United Kingdom
freq,1350,5829,5918,13097,981330


In [7]:
# Date range
print(f"Period: {df['InvoiceDate'].min().date()} ->  {df['InvoiceDate'].max().date()}")
print(f"Unique invoices:   {df['Invoice'].nunique():,}")
print(f"Unique customers:  {df['Customer ID'].nunique():,}")
print(f"Unique products:   {df['StockCode'].nunique():,}")
print(f"Unique countries:  {df['Country'].nunique():,}")

Period: 2009-12-01 ->  2011-12-09
Unique invoices:   53,628
Unique customers:  5,942
Unique products:   5,305
Unique countries:  43


---

## 4. Data Quality Assessment <a id="4-data-quality-assessment"></a>

### 4.1 Missing Values

In [8]:
# Missing values per column


### 4.2 Cancellations

In [9]:
# Isolate cancellations (Invoice starts with 'C')
# How many? What % of total? What do they look like?


### 4.3 Invalid Prices

In [10]:
# Prices <= 0: how many? Any pattern in StockCode or Description?


### 4.4 Negative Quantities Outside Cancellations

In [11]:
# Non-cancellation rows with Quantity < 0
# Do they cluster around specific StockCodes, dates, or customers?


---

## 5. Univariate Analysis <a id="5-univariate-analysis"></a>

### 5.1 Quantity

In [12]:
# Distribution of Quantity (exclude cancellations)
# Watch for extreme outliers


### 5.2 Price

In [13]:
# Distribution of Price (valid rows only)
# What is the typical price range? Any extreme values?


### 5.3 Country

In [14]:
# Transactions per country — top 15
# Is the dataset heavily skewed toward one country (UK)?


### 5.4 Customer ID

In [15]:
# Proportion of transactions with vs without Customer ID
# Guest checkouts vs identified customers


---

## 6. Temporal Analysis <a id="6-temporal-analysis"></a>

### 6.1 Monthly Revenue

In [16]:
# Create Revenue column: Quantity * Price
# Aggregate by month and plot
# Look for seasonality and anomalies


### 6.2 Daily and Weekly Patterns

In [17]:
# Transactions by day of week and hour of day
# When do customers shop most?


---

## 7. Customer Analysis <a id="7-customer-analysis"></a>

### 7.1 Purchases per Customer

In [18]:
# Distribution of invoice count per customer
# One-time buyers vs repeat customers


### 7.2 Revenue per Customer

In [19]:
# Distribution of total revenue per customer
# Is revenue concentrated in a small group? (Pareto principle)


### 7.3 Top Customers

In [20]:
# Top 10 customers by revenue


---

## 8. Product Analysis <a id="8-product-analysis"></a>

### 8.1 Top Products by Revenue

In [21]:
# Top 15 products by total revenue


### 8.2 Top Products by Frequency

In [22]:
# Top 15 products by number of transactions


---

## 9. Conclusions & Cleaning Decisions <a id="9-conclusions--cleaning-decisions"></a>

### Key Findings

| # | Finding | Impact |
|---|---------|--------|
| 1 | | |
| 2 | | |
| 3 | | |

### Cleaning Decisions for `cleaning.py`

Based on this EDA, the following decisions will guide `src/features/cleaning.py`:

- **Cancellations:** 
- **Invalid prices:** 
- **Negative quantities:** 
- **Missing Customer ID:** 
- **Outliers:** 

### Next Step

-> `src/features/cleaning.py` (implement the decisions above)  
-> `notebooks/02_rfm_segmentation.ipynb` (RFM analysis on clean data)

---

### References

- Chen, D., Sain, S.L., Guo, K. (2012). Data mining for the online retail industry. *Journal of Database Marketing & Customer Strategy Management*, 19(3), 197–208.
- [UCI ML Repository — Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii)

### Versioning

- Version: 0.1.0  
- Last updated: 05/22/2026